In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
car = pd.read_csv(r"C:\Users\PIYA\OneDrive\Desktop\car data.csv")

In [ ]:
car

In [ ]:
print(car['Selling_type'].unique())
print(car['Fuel_Type'].unique())
print(car['Transmission'].unique())
print(car['Owner'].unique())

In [ ]:
car.describe()

In [ ]:
car.info()

In [ ]:

car.duplicated().sum()

In [ ]:
car.drop_duplicates(inplace= True)

In [ ]:
car.isnull().sum()

In [ ]:
#converting the dtype of year
car["Year"]= pd.to_datetime(car["Year"], format = '%Y').dt.year

In [ ]:
car["Owner"] = car["Owner"].astype("int32")
car["Driven_kms"] = car["Driven_kms"].astype("int32")

In [ ]:
car.info()

In [ ]:
car["Year"].unique()

In [ ]:
car["Year"].nunique()

In [ ]:
#SELECTING IMPORTANT DATA FOR MODEL BUILDING(#FEATURE SELECTION)
car = car.drop(columns= "Car_Name")

In [ ]:
### ADDING AGE COLUMN OF CAR
car["current year"]= 2023

In [ ]:

car['Age of car']= car["current year"]-car["Year"]

In [ ]:
car

In [ ]:
car = car.drop(columns= ["current year", "Year"])
car

In [ ]:
car = pd.get_dummies(data=car,  drop_first= True)
car

In [ ]:
g= ['Fuel_Type_Diesel', 'Fuel_Type_Petrol', 'Selling_type_Individual', 'Transmission_Manual']
car[g]= car[g].astype('int')
car.head(3)

In [ ]:
car.corr()

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.pairplot(car)

In [ ]:
sns.heatmap(car.corr(), annot= True, cmap= 'Reds')

In [ ]:
y = car['Selling_Price'] #DEPENDENT VARIABLE AND TARGET
x = car.drop(columns= ['Selling_Price']) # INPUT AND INDEPENDENT DATA

In [ ]:
y

In [ ]:
x

In [ ]:
x['Owner'].unique()

In [ ]:
### Feature Importance

from sklearn.ensemble import ExtraTreesRegressor
model = ExtraTreesRegressor()
model.fit(x,y)

In [ ]:
print(model.feature_importances_)

In [ ]:
#plot graph of feature importances for better visualization
feat_importances = pd.Series(model.feature_importances_, index=x.columns)
feat_importances.nlargest(5).plot(kind='barh')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=5)

In [ ]:
x_train.shape

In [ ]:
x_test.shape

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Creating a RandomForestRegressor without specifying the 'criterion' parameter
regressor = RandomForestRegressor()

# Fit the model
regressor.fit(x_train, y_train)

In [ ]:
ypred = regressor.predict(x_test)
ypred

In [ ]:
from sklearn.metrics import r2_score
r2_score(y_test, ypred)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
parameters = {
    'n_estimators': [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000],
    'criterion': ['squared_error', 'absolute_error', 'poisson', 'friedman_mse'],  # Use valid criterion values
    'max_depth': [10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10, 20, 50],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['auto', 'sqrt', 'log2']
}

In [ ]:
parameters

In [ ]:
random_cv = RandomizedSearchCV(estimator=regressor, param_distributions=parameters, n_iter=10, 
                               scoring ='neg_mean_absolute_error',random_state=42, cv=5, verbose=2, n_jobs=-1)

In [ ]:
random_cv.fit(x_train, y_train)

In [ ]:

random_cv.best_params_

In [ ]:
random_cv.best_score_

In [ ]:
predictions=random_cv.predict(x_test)

In [ ]:
sns.distplot(y_test-predictions)

In [ ]:
plt.scatter(y_test, predictions)

In [ ]:
from sklearn import metrics

In [ ]:
print('MAE:', metrics.mean_absolute_error(y_test, predictions))
print('MSE:', metrics.mean_squared_error(y_test, predictions))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_test, predictions)))

In [ ]:
### predicting a singe observation
x.head(3)

In [ ]:
single_ob = np.array([9.5, 15000, 0.0, 5.0, 1.0, 0.0, 0.0, 1.0])
single_ob = single_ob.reshape(1, -1)

In [ ]:
regressor.predict(single_ob)

In [ ]:
car.head()